# Clipt — Roboflow Model Training v2

Trains **9 models** across the 3 sports Clipt supports:

| Priority | Sport | Models | Total Images |
|----------|-------|--------|--------------|
| 1 | Basketball | jersey_number_v2, jersey_number_v3, player_detector | 11,945 |
| 2 | Football | positions_detector, presnap_detector, universal_v1, universal_v2 | 2,965 |
| 3 | Lacrosse | detector_v1, detector_v2 | 908 |

## Why v2?
The v1 `basketball_jersey_ocr.pt` scored **mAP50: 0.10** — completely unusable.
This notebook replaces it with better datasets and adds lacrosse support.

## Prerequisites
- **Runtime: T4 GPU** (Runtime → Change runtime type → T4 GPU)
- **Roboflow API key** stored in Colab Secrets as `ROBOFLOW_API_KEY`
- Training takes ~3-4 hours total on T4

## After Training
Only models with **mAP50 >= 0.5** are worth deploying.
Download them and place in `jersey-detection/app/model/`.
The `roboflow_detector.py` already has stub loaders for all 9 models.

## Section 1 — Setup

In [ ]:
import os

# Read API key from Colab Secrets (key icon in sidebar)
from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')

!pip install roboflow ultralytics -q
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key=api_key)

# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU! Training will be extremely slow.")

## Section 2 — Download ALL Datasets

### PRIORITY 1: Basketball (replace failed 0.10 mAP model)

Three datasets to give us the best possible basketball jersey detection:
- **6,932 images** from VolleyAi (multi-sport digit detection, largest available)
- **3,615 images** from Roboflow official (NBA Playoffs footage, basketball-specific)
- **1,398 images** for basketball player detection (enables crop-then-OCR pipeline)

In [ ]:
# Dataset 1 — VolleyAi Jersey Number Detection (6,932 images)
# Largest open-source jersey number dataset — works across all sports
# Classes: jersey number digits | License: CC BY 4.0
# Output model: basketball_jersey_number_v2.pt (PRIMARY basketball model)
project_1 = rf.workspace("volleyai-actions").project("jersey-number-detection-s01j4")
dataset_1 = project_1.version(2).download("yolov8")
print(f"Dataset 1 location: {dataset_1.location}")

In [ ]:
# Dataset 2 — Roboflow Basketball Jersey Numbers OCR v5 (3,615 images)
# Collected from 2025 NBA Playoffs — basketball-specific jersey crops
# Classes: jersey number OCR | License: CC BY 4.0
# Output model: basketball_jersey_number_v3.pt
project_2 = rf.workspace("roboflow-jvuqo").project("basketball-jersey-numbers-ocr")
dataset_2 = project_2.version(5).download("yolov8")
print(f"Dataset 2 location: {dataset_2.location}")

In [ ]:
# Dataset 3 — Roboflow Basketball Player Detection 2 (1,398 images)
# Player bounding boxes — enables the crop-then-OCR pipeline for basketball
# Classes: players | License: CC BY 4.0
# Output model: basketball_player_detector.pt
project_3 = rf.workspace("roboflow-jvuqo").project("basketball-player-detection-2")
dataset_3 = project_3.version(1).download("yolov8")
print(f"Dataset 3 location: {dataset_3.location}")

### PRIORITY 2: Football (additional data on top of existing v1 models)

We already have `football_digit_detector.pt`, `football_player_detector.pt`, and
`football_jersey_tracker.pt` from v1 training. These datasets add:
- **Position classification** (QB, WR, RB, DB) — can auto-detect Dustin's position
- **Presnap formation** analysis
- **Universal jersey numbers** that work as a fallback for any sport

In [ ]:
# Dataset 4 — bronkscottema Football Players by Position (755 images)
# American football with position-level classes
# Classes: CENTER, DB, LB, QB, RB, S, SKILL, WR (8 classes)
# License: CC BY 4.0 | v15 reported 97.3% mAP
# Output model: football_positions_detector.pt
project_4 = rf.workspace("bronkscottema").project("football-players-zm06l")
dataset_4 = project_4.version(15).download("yolov8")
print(f"Dataset 4 location: {dataset_4.location}")

In [ ]:
# Dataset 5 — Football Presnap Tracker (828 images)
# Presnap formation analysis — useful for play classification
# Classes: football players in formation | License: CC BY 4.0
# Output model: football_presnap_detector.pt
project_5 = rf.workspace("football-tracking").project("football-presnap-tracker")
dataset_5 = project_5.version(1).download("yolov8")
print(f"Dataset 5 location: {dataset_5.location}")

In [ ]:
# Dataset 6 — Dark Blue JerseyNumbers (826 images)
# Multi-sport jersey number detection — universal fallback
# Classes: number classes | License: CC BY 4.0
# Output model: jersey_number_universal_v1.pt
project_6 = rf.workspace("dark-blue-jt0mg").project("jerseynumbers")
dataset_6 = project_6.version(5).download("yolov8")
print(f"Dataset 6 location: {dataset_6.location}")

In [ ]:
# Dataset 7 — yakovk Jersey Numbers (556 images)
# General sports jersey number detection — second universal fallback
# Classes: player-jersey-numbers | License: CC BY 4.0
# Output model: jersey_number_universal_v2.pt
project_7 = rf.workspace("yakovk").project("jersey-numbers-i1wn5")
dataset_7 = project_7.version(1).download("yolov8")
print(f"Dataset 7 location: {dataset_7.location}")

### PRIORITY 3: Lacrosse

Clipt supports lacrosse athletes. These are the only lacrosse datasets on Roboflow Universe.
Both are small (<530 images) so we train with extra epochs and heavy augmentation.

In [ ]:
# Dataset 8 — RySEAI Lacrosse Object Detection (528 images)
# Best available lacrosse dataset
# Classes: Goalie, Longpole, Referee, Shortstick, sports ball
# License: CC BY 4.0
# Output model: lacrosse_detector_v1.pt
project_8 = rf.workspace("ryseai").project("lacrosse-object-detection")
dataset_8 = project_8.version(1).download("yolov8")
print(f"Dataset 8 location: {dataset_8.location}")

In [ ]:
# Dataset 9 — Sports Computer Vision / Lacrosse (380 images)
# Additional lacrosse data to supplement dataset 8
# Classes: Goalie, Lacrosse Ball, Long stick, Referee, Short stick
# License: CC BY 4.0
# Output model: lacrosse_detector_v2.pt
project_9 = rf.workspace("computer-vision-ho8xk").project("sports-computer-vision")
dataset_9 = project_9.version(1).download("yolov8")
print(f"Dataset 9 location: {dataset_9.location}")

## Section 3 — Train ALL 9 Models

Training settings by dataset size:
- **Large (1000+ images):** epochs=60, imgsz=640, augment=True
- **Medium (500-999 images):** epochs=80, imgsz=640, augment=True
- **Small (<500 images):** epochs=100, imgsz=640, augment=True, extra augmentation (hsv_s=0.9, degrees=10, scale=0.6)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 1: basketball_jersey_number_v2.pt
# Dataset: VolleyAi (6,932 images) — PRIMARY basketball jersey model
# This replaces the failed basketball_jersey_ocr.pt (mAP50: 0.10)
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: basketball_jersey_number_v2 (6,932 images)")
print("=" * 60)

model_1 = YOLO("yolov8n.pt")
model_1.train(
    data=f"{dataset_1.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="basketball_jersey_number_v2",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)
print("basketball_jersey_number_v2 training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 2: basketball_jersey_number_v3.pt
# Dataset: Roboflow Basketball OCR (3,615 images) — NBA Playoffs
# Basketball-specific — should outperform the generic v2 on NBA footage
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: basketball_jersey_number_v3 (3,615 images)")
print("=" * 60)

model_2 = YOLO("yolov8n.pt")
model_2.train(
    data=f"{dataset_2.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="basketball_jersey_number_v3",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)
print("basketball_jersey_number_v3 training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 3: basketball_player_detector.pt
# Dataset: Basketball Player Detection 2 (1,398 images)
# Enables crop-then-OCR: detect player boxes, crop, then run digit models
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: basketball_player_detector (1,398 images)")
print("=" * 60)

model_3 = YOLO("yolov8n.pt")
model_3.train(
    data=f"{dataset_3.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    name="basketball_player_detector",
    patience=10,
    device=0,
    augment=True,
)
print("basketball_player_detector training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 4: football_positions_detector.pt
# Dataset: bronkscottema Football Positions (755 images)
# Detects QB, WR, RB, DB, LB, CENTER, S, SKILL positions
# Can auto-identify Dustin's position from game film
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: football_positions_detector (755 images)")
print("=" * 60)

model_4 = YOLO("yolov8n.pt")
model_4.train(
    data=f"{dataset_4.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    name="football_positions_detector",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print("football_positions_detector training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 5: football_presnap_detector.pt
# Dataset: Football Presnap Tracker (828 images)
# Presnap formation detection — helps classify play types
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: football_presnap_detector (828 images)")
print("=" * 60)

model_5 = YOLO("yolov8n.pt")
model_5.train(
    data=f"{dataset_5.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    name="football_presnap_detector",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print("football_presnap_detector training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 6: jersey_number_universal_v1.pt
# Dataset: Dark Blue JerseyNumbers (826 images)
# Multi-sport jersey number fallback — works for any sport
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: jersey_number_universal_v1 (826 images)")
print("=" * 60)

model_6 = YOLO("yolov8n.pt")
model_6.train(
    data=f"{dataset_6.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    name="jersey_number_universal_v1",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print("jersey_number_universal_v1 training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 7: jersey_number_universal_v2.pt
# Dataset: yakovk Jersey Numbers (556 images)
# Second universal fallback — different annotation style
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: jersey_number_universal_v2 (556 images)")
print("=" * 60)

model_7 = YOLO("yolov8n.pt")
model_7.train(
    data=f"{dataset_7.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    name="jersey_number_universal_v2",
    patience=15,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print("jersey_number_universal_v2 training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 8: lacrosse_detector_v1.pt
# Dataset: RySEAI Lacrosse (528 images) — SMALL, extra augmentation
# Detects: Goalie, Longpole (defense), Shortstick (offense), Referee
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: lacrosse_detector_v1 (528 images)")
print("=" * 60)

model_8 = YOLO("yolov8n.pt")
model_8.train(
    data=f"{dataset_8.location}/data.yaml",
    epochs=100,  # Small dataset — extra epochs
    imgsz=640,
    batch=16,
    name="lacrosse_detector_v1",
    patience=20,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.9,   # Extra color augmentation for small dataset
    hsv_v=0.4,
    degrees=10.0, # Extra rotation
    translate=0.15,
    scale=0.6,    # Extra scale jitter
    fliplr=0.5,
    mosaic=1.0,
)
print("lacrosse_detector_v1 training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Model 9: lacrosse_detector_v2.pt
# Dataset: Sports CV Lacrosse (380 images) — SMALL, extra augmentation
# Different annotation source — diversity helps
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: lacrosse_detector_v2 (380 images)")
print("=" * 60)

model_9 = YOLO("yolov8n.pt")
model_9.train(
    data=f"{dataset_9.location}/data.yaml",
    epochs=100,  # Small dataset — extra epochs
    imgsz=640,
    batch=16,
    name="lacrosse_detector_v2",
    patience=20,
    device=0,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.9,   # Extra color augmentation for small dataset
    hsv_v=0.4,
    degrees=10.0, # Extra rotation
    translate=0.15,
    scale=0.6,    # Extra scale jitter
    fliplr=0.5,
    mosaic=1.0,
)
print("lacrosse_detector_v2 training complete!")

## Section 4 — Validate All Models

Print mAP50 for every model. **Skip any with mAP50 < 0.5** — not worth deploying.

In [ ]:
print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

MODELS = {
    "basketball_jersey_number_v2": "runs/detect/basketball_jersey_number_v2/weights/best.pt",
    "basketball_jersey_number_v3": "runs/detect/basketball_jersey_number_v3/weights/best.pt",
    "basketball_player_detector": "runs/detect/basketball_player_detector/weights/best.pt",
    "football_positions_detector": "runs/detect/football_positions_detector/weights/best.pt",
    "football_presnap_detector": "runs/detect/football_presnap_detector/weights/best.pt",
    "jersey_number_universal_v1": "runs/detect/jersey_number_universal_v1/weights/best.pt",
    "jersey_number_universal_v2": "runs/detect/jersey_number_universal_v2/weights/best.pt",
    "lacrosse_detector_v1": "runs/detect/lacrosse_detector_v1/weights/best.pt",
    "lacrosse_detector_v2": "runs/detect/lacrosse_detector_v2/weights/best.pt",
}

deployable = []
skipped = []

for name, path in MODELS.items():
    if os.path.exists(path):
        model = YOLO(path)
        metrics = model.val()
        map50 = metrics.box.map50
        map50_95 = metrics.box.map
        verdict = "DEPLOY" if map50 >= 0.5 else "SKIP"
        if map50 >= 0.5:
            deployable.append((name, path, map50))
        else:
            skipped.append((name, map50))
        print(f"  {name}: mAP50={map50:.3f}  mAP50-95={map50_95:.3f}  [{verdict}]")
    else:
        print(f"  {name}: MISSING (training failed?)")

print("\n" + "-" * 60)
print(f"DEPLOYABLE: {len(deployable)}/9 models")
for n, _, s in deployable:
    print(f"  + {n}.pt  (mAP50: {s:.3f})")
if skipped:
    print(f"\nSKIPPED (mAP50 < 0.5): {len(skipped)} models")
    for n, s in skipped:
        print(f"  - {n}.pt  (mAP50: {s:.3f})")
print("=" * 60)

## Section 5 — Download Deployable Models

Only downloads models with mAP50 >= 0.5. Place them in `jersey-detection/app/model/`.

In [ ]:
from google.colab import files
import shutil

print("Downloading deployable models...\n")
downloaded = 0

for name, path, score in deployable:
    output_name = f"{name}.pt"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        shutil.copy(path, output_name)
        files.download(output_name)
        print(f"  + {output_name}  ({size_mb:.1f} MB, mAP50: {score:.3f})")
        downloaded += 1

print(f"\n{'=' * 60}")
print(f"Downloaded {downloaded} models.")
print(f"")
print(f"Next steps:")
print(f"  1. Place .pt files in jersey-detection/app/model/")
print(f"  2. Add !app/model/{{filename}}.pt lines to .gitignore")
print(f"  3. git add app/model/*.pt && git commit && git push")
print(f"  4. Railway auto-deploys — roboflow_detector.py already")
print(f"     has stub loaders for all 9 model names.")
print(f"{'=' * 60}")